In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity


# 1. LOAD TOKENIZER — converts text to integer IDs (no semantics)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")


In [ ]:
# 2. TOKENIZE — chop text into IDs, pad/truncate to 512 tokens
encoded = tokenizer(
    faq_texts,
    padding="max_length",
    truncation=True,
    max_length=512,
    return_tensors="pt",
)
# Shape: [num_samples, 512] — every FAQ padded or truncated to equal length



In [ ]:
# 3. LOAD EMBEDDING MODEL — 4-bit quantized for memory efficiency

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
model = AutoModel.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",  # ~2.5 GB VRAM instead of ~16 GB
)


In [ ]:
# 4. MEAN POOLING — average token vectors into one document vector
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked = last_hidden_state * mask
    summed = masked.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts  # Shape: [batch_size, 1536]


In [ ]:
# 5. GENERATE EMBEDDINGS — batch processing to avoid OOM
with torch.no_grad():
    for i in range(0, num_samples, 4):
        outputs = model(input_ids=batch_ids, attention_mask=batch_mask)
        batch_emb = mean_pool(outputs.last_hidden_state, batch_mask)
        all_embeddings.append(batch_emb.cpu())

embeddings = torch.cat(all_embeddings, dim=0).numpy()  # Shape: [15, 1536]


In [ ]:
# 6. VALIDATE — cosine similarity within categories

for category in ["Contracts", "NDAs", "IP"]:
    idx = df[df["category"] == category].index
    sim = cosine_similarity(embeddings[idx])
    mean_sim = (sim.sum() - len(idx)) / (len(idx) * (len(idx) - 1))
    print(f"{category}: within-group similarity = {mean_sim:.4f}")